In [26]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import seaborn as sns
sns.set()
import umap

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.decomposition import IncrementalPCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN
from sklearn.cluster import HDBSCAN
from sklearn.manifold import TSNE

from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import rand_score
from sklearn.metrics import normalized_mutual_info_score
from mpl_toolkits.mplot3d import Axes3D 

In [27]:
file = "./data/weather/tokens_train.pt"
data = torch.load(file, weights_only=False, map_location="cpu", mmap=True)
print(data.keys())
#print(data)

dict_keys(['tokens', 'timestamps', 'hours', 'doys', 'latent_shape', 'num_channels', 'split'])


In [28]:
print(data["tokens"].shape, "Data type: ",data["tokens"].dtype)
print(data["timestamps"])
print("Latent Shape: ",data["latent_shape"])
print("Number of Channels: ",data["num_channels"])

torch.Size([59900, 1024, 8, 15]) Data type:  torch.float16
['1979-01-01T00:00:00' '1979-01-01T06:00:00' '1979-01-01T12:00:00' ...
 '2019-12-31T06:00:00' '2019-12-31T12:00:00' '2019-12-31T18:00:00']
Latent Shape:  [1024, 8, 15]
Number of Channels:  1024


In [29]:
hours = data["hours"].numpy()
N = len(hours)
keep = np.where(hours == 0)[0] 

output = {}
for k, v in data.items():
    if hasattr(v, "shape") and len(v.shape) > 0 and v.shape[0] == N:   
        output[k] = v[torch.from_numpy(keep)] if torch.is_tensor(v) else v[keep]
    else:
        output[k] = v 

torch.save(output, "tokens_train_daily.pt")
print(len(keep), "of", N)  

14975 of 59900


In [30]:
daily = torch.load("./tokens_train_daily.pt", weights_only=False, map_location="cpu", mmap=True)
#print(daily)
timestamp = daily["timestamps"]
print(timestamp)

['1979-01-01T00:00:00' '1979-01-02T00:00:00' '1979-01-03T00:00:00' ...
 '2019-12-29T00:00:00' '2019-12-30T00:00:00' '2019-12-31T00:00:00']


In [31]:
token = daily["tokens"]
N = token.shape[0]
D = 1024*8*15

z = np.empty((N,D),np.float16)
for i in range(0,N,1000):
    z[i:i+1000] = token[i:i+1000].float().reshape(-1, D).numpy()

print(z.shape)

(14975, 122880)
